# Crop Yield Prediction Using Machine Learning

**BTech Machine Learning Lab Project**

**Team Members:** Nidhish Bansal and Yash Vishnoi

Agriculture is central to food security and the global economy. This notebook studies whether historical country, crop, weather, and pesticide information can be used to estimate crop yield. It demonstrates a complete supervised-learning workflow: data inspection, exploratory analysis, leak-safe preprocessing, model comparison, hyperparameter tuning, evaluation, and interpretation.

The target is `hg/ha_yield` (hectograms per hectare). For context, **tonnes/hectare = hg/ha ÷ 10,000**. The notebook is an educational scenario model, not a weather forecast or a production agricultural advisory system.

## 1. Imports and reproducibility

A fixed random seed makes the random split and tree models reproducible.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 20)

## 2. Load and understand the data

The merged dataset contains country (`Area`), crop (`Item`), year, annual rainfall, pesticide use, average temperature, and recorded yield.

In [ ]:
DATA_PATH = Path('dataset/yield_df.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected dataset at: {DATA_PATH.resolve()}')

raw_df = pd.read_csv(DATA_PATH)
print(f'Raw dataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
display(raw_df.head())
raw_df.info()

In [ ]:
quality_summary = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing': raw_df.isna().sum(),
    'unique_values': raw_df.nunique()
})
display(quality_summary)
print(f'Exact duplicate rows: {raw_df.duplicated().sum():,}')
display(raw_df.describe(include='all').T)

### Cleaning decisions

- Remove `Unnamed: 0`, which is only a saved CSV row index.
- Verify that required fields are present and contain no missing values.
- Do not delete unusually high or low yields automatically: different crops naturally have very different yield scales, and there is no evidence that these observations are errors.
- Keep `Year` as a numeric predictor and country/crop as categorical predictors.

In [ ]:
df = raw_df.drop(columns=['Unnamed: 0'], errors='ignore').copy()
required_columns = {
    'Area', 'Item', 'Year', 'hg/ha_yield',
    'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp'
}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'
assert not df[list(required_columns)].isna().any().any(), 'Required data contains missing values.'
assert (df['hg/ha_yield'] > 0).all(), 'Yield must be positive.'

print(f'Clean dataset: {len(df):,} rows, {df.Area.nunique()} countries, {df.Item.nunique()} crops')
print(f'Year range: {df.Year.min()}–{df.Year.max()}')
print('Crops:', ', '.join(sorted(df.Item.unique())))

## 3. Exploratory data analysis

The upper-left histogram is limited to the 99th percentile only for readability; no observations are removed from modelling. Crop-wise medians are used because the target is strongly right-skewed.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

yield_99 = df['hg/ha_yield'].quantile(0.99)
sns.histplot(df.loc[df['hg/ha_yield'] <= yield_99, 'hg/ha_yield'], bins=40, ax=axes[0, 0], color='#2a9d8f')
axes[0, 0].set(title='Yield distribution (up to 99th percentile)', xlabel='Yield (hg/ha)')

crop_medians = df.groupby('Item')['hg/ha_yield'].median().sort_values()
sns.barplot(x=crop_medians.values, y=crop_medians.index, ax=axes[0, 1], color='#457b9d')
axes[0, 1].set(title='Median yield by crop', xlabel='Median yield (hg/ha)', ylabel='Crop')

yearly_yield = df.groupby('Year', as_index=False)['hg/ha_yield'].mean()
sns.lineplot(data=yearly_yield, x='Year', y='hg/ha_yield', marker='o', ax=axes[1, 0], color='#e76f51')
axes[1, 0].set(title='Average recorded yield over time', ylabel='Mean yield (hg/ha)')

top_countries = df['Area'].value_counts().head(12).sort_values()
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[1, 1], color='#f4a261')
axes[1, 1].set(title='Countries with the most observations', xlabel='Number of rows', ylabel='Country')

plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = [
    'Year', 'hg/ha_yield', 'average_rain_fall_mm_per_year',
    'pesticides_tonnes', 'avg_temp'
]
plt.figure(figsize=(9, 6))
sns.heatmap(df[numeric_columns].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation between numeric variables')
plt.tight_layout()
plt.show()

display(df.groupby('Item')['hg/ha_yield'].agg(['count', 'mean', 'median', 'min', 'max']).round(1))

## 4. Features, target, and evaluation designs

Two test designs answer different questions:

1. **Random 80/20 split:** estimates performance on observations drawn from the same historical mixture.
2. **Time-based split:** trains on 1990–2009 and tests on 2010–2013, which is closer to predicting future years.

The time split is the more important result for the project claim. The test data is never used to fit preprocessing or tune hyperparameters.

In [ ]:
TARGET = 'hg/ha_yield'
CATEGORICAL_FEATURES = ['Area', 'Item']
NUMERIC_FEATURES = [
    'Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp'
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_random_train, X_random_test, y_random_train, y_random_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

time_train_mask = df['Year'] <= 2009
time_test_mask = df['Year'] >= 2010
X_time_train = X.loc[time_train_mask].reset_index(drop=True)
y_time_train = y.loc[time_train_mask].reset_index(drop=True)
X_time_test = X.loc[time_test_mask].reset_index(drop=True)
y_time_test = y.loc[time_test_mask].reset_index(drop=True)

unseen_countries = sorted(set(X_time_test['Area']) - set(X_time_train['Area']))
print(f'Random split: {len(X_random_train):,} train / {len(X_random_test):,} test')
print(f'Time split:   {len(X_time_train):,} train / {len(X_time_test):,} test')
print(f'Time-test years: {X_time_test.Year.min()}–{X_time_test.Year.max()}')
print(f'Countries first seen in time test: {unseen_countries or "None"}')

## 5. Leak-safe preprocessing and models

`OneHotEncoder(handle_unknown='ignore')` converts country and crop names to numeric indicators and safely handles a country that appears only in the future test period. Numeric features are standardized. Each estimator is placed after preprocessing in one pipeline, so transformations are fitted only on the relevant training data.

The models represent increasing nonlinear capacity: Linear Regression (baseline), a single Decision Tree, and a Random Forest ensemble.

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
            ('numeric', StandardScaler(), NUMERIC_FEATURES),
        ]
    )


def make_pipeline(estimator):
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('model', estimator),
    ])

## 6. Random Forest tuning with expanding-year validation

Only the strongest nonlinear candidate is tuned, keeping the experiment compact. The validation folds move forward through time and remain entirely inside the 1990–2009 training period. The final 2010–2013 test period stays untouched.

In [ ]:
# (training end year, validation start year, validation end year)
fold_years = [(2003, 2004, 2005), (2005, 2006, 2007), (2007, 2008, 2009)]
expanding_year_cv = []
for train_end, valid_start, valid_end in fold_years:
    train_idx = np.flatnonzero(X_time_train['Year'].to_numpy() <= train_end)
    valid_idx = np.flatnonzero(X_time_train['Year'].between(valid_start, valid_end).to_numpy())
    assert len(train_idx) and len(valid_idx)
    expanding_year_cv.append((train_idx, valid_idx))
    print(f'Train through {train_end}: {len(train_idx):>5,} rows | '          f'validate {valid_start}–{valid_end}: {len(valid_idx):>4,} rows')

rf_search_pipeline = make_pipeline(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
)
rf_param_grid = [
    {'model__n_estimators': [20], 'model__max_depth': [None], 'model__min_samples_split': [2]},
    {'model__n_estimators': [50], 'model__max_depth': [None], 'model__min_samples_split': [5]},
    {'model__n_estimators': [20], 'model__max_depth': [20],   'model__min_samples_split': [5]},
    {'model__n_estimators': [50], 'model__max_depth': [20],   'model__min_samples_split': [2]},
]

rf_grid_search = GridSearchCV(
    estimator=rf_search_pipeline,
    param_grid=rf_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=expanding_year_cv,
    n_jobs=1,  # Reliable inside Jupyter; the compact grid keeps runtime manageable.
    refit=True,
    return_train_score=False,
    error_score='raise',
)
rf_grid_search.fit(X_time_train, y_time_train)

best_rf_params = {key.replace('model__', ''): value for key, value in rf_grid_search.best_params_.items()}
print('Best Random Forest parameters:', best_rf_params)
print(f'Best validation RMSE: {-rf_grid_search.best_score_:,.2f} hg/ha')

In [ ]:
cv_results = pd.DataFrame(rf_grid_search.cv_results_)[
    ['param_model__n_estimators', 'param_model__max_depth',
     'param_model__min_samples_split', 'mean_test_score', 'rank_test_score']
].copy()
cv_results['validation_RMSE'] = -cv_results.pop('mean_test_score')
display(cv_results.sort_values('rank_test_score').reset_index(drop=True))

## 7. Model comparison on both test designs

MAE is the typical absolute error, RMSE penalizes large errors more strongly, and R² measures explained variation. Lower MAE/RMSE and higher R² are better. Train R² is included to reveal overfitting, especially for a single decision tree.

In [ ]:
def get_models():
    return {
        'Linear Regression': LinearRegression(),
        'Decision Tree': DecisionTreeRegressor(random_state=RANDOM_STATE),
        'Random Forest (tuned)': RandomForestRegressor(
            **best_rf_params, random_state=RANDOM_STATE, n_jobs=1
        ),
    }


def evaluate_models(split_name, X_train, X_test, y_train, y_test):
    rows, fitted_models, predictions = [], {}, {}
    # Each call creates fresh model objects, so the two experiments stay independent.
    for model_name, model in get_models().items():
        pipeline = make_pipeline(model)
        pipeline.fit(X_train, y_train)
        train_prediction = pipeline.predict(X_train)
        test_prediction = pipeline.predict(X_test)

        rows.append({
            'Split': split_name,
            'Model': model_name,
            'MAE': mean_absolute_error(y_test, test_prediction),
            'RMSE': mean_squared_error(y_test, test_prediction) ** 0.5,
            'Test R2': r2_score(y_test, test_prediction),
            'Train R2': r2_score(y_train, train_prediction),
        })
        fitted_models[model_name] = pipeline
        predictions[model_name] = test_prediction
    return rows, fitted_models, predictions


random_rows, random_models, random_predictions = evaluate_models(
    'Random 80/20', X_random_train, X_random_test, y_random_train, y_random_test
)
time_rows, time_models, time_predictions = evaluate_models(
    'Time 2010–2013', X_time_train, X_time_test, y_time_train, y_time_test
)

results = pd.DataFrame(random_rows + time_rows)
results[['MAE', 'RMSE', 'Test R2', 'Train R2']] = results[
    ['MAE', 'RMSE', 'Test R2', 'Train R2']
].round(3)
display(results.sort_values(['Split', 'RMSE']).reset_index(drop=True))

assert np.isfinite(results[['MAE', 'RMSE', 'Test R2', 'Train R2']].to_numpy()).all()
assert set(results['Model']) == {'Linear Regression', 'Decision Tree', 'Random Forest (tuned)'}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'Test R2']):
    sns.barplot(data=results, x='Model', y=metric, hue='Split', ax=ax)
    ax.set_title(f'{metric} comparison')
    ax.tick_params(axis='x', rotation=20)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 8. Diagnose the strongest time-split model

The plots below focus on the lowest-RMSE model on the untouched future test set. Points near the diagonal indicate accurate predictions. A good residual plot should be centered near zero without a strong pattern.

In [ ]:
time_result_table = results[results['Split'] == 'Time 2010–2013']
best_time_model_name = time_result_table.loc[time_result_table['RMSE'].idxmin(), 'Model']
best_time_model = time_models[best_time_model_name]
best_time_prediction = time_predictions[best_time_model_name]
residuals = y_time_test.to_numpy() - best_time_prediction

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=y_time_test, y=best_time_prediction, alpha=0.35, s=25, ax=axes[0])
limits = [min(y_time_test.min(), best_time_prediction.min()),
          max(y_time_test.max(), best_time_prediction.max())]
axes[0].plot(limits, limits, '--', color='black', linewidth=1.5)
axes[0].set(title=f'Actual vs predicted: {best_time_model_name}',
            xlabel='Actual yield (hg/ha)', ylabel='Predicted yield (hg/ha)')

sns.scatterplot(x=best_time_prediction, y=residuals, alpha=0.35, s=25, ax=axes[1])
axes[1].axhline(0, linestyle='--', color='black', linewidth=1.5)
axes[1].set(title='Residual plot', xlabel='Predicted yield (hg/ha)',
            ylabel='Actual − predicted (hg/ha)')
plt.tight_layout()
plt.show()

## 9. Permutation feature importance

Permutation importance measures how much future-test RMSE worsens when one original input column is shuffled. It is model-specific and describes predictive usefulness, not causation.

In [ ]:
importance = permutation_importance(
    best_time_model, X_time_test, y_time_test,
    scoring='neg_root_mean_squared_error', n_repeats=3,
    random_state=RANDOM_STATE, n_jobs=1
)
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': importance.importances_mean,
    'Std': importance.importances_std,
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'],
         xerr=importance_df['Std'], color='#457b9d', alpha=0.9)
plt.xlabel('Increase in RMSE after shuffling (hg/ha)')
plt.title(f'Permutation importance: {best_time_model_name}')
plt.tight_layout()
plt.show()
display(importance_df.sort_values('Importance', ascending=False).reset_index(drop=True))

## 10. Conclusions and limitations

In [ ]:
best_random_row = results.loc[results[results['Split'] == 'Random 80/20']['RMSE'].idxmin()]
best_time_row = results.loc[results[results['Split'] == 'Time 2010–2013']['RMSE'].idxmin()]
time_penalty = best_time_row['RMSE'] - best_random_row['RMSE']

display(Markdown(f'''
### Main findings

- On the random split, **{best_random_row['Model']}** achieved the lowest RMSE: **{best_random_row['RMSE']:,.2f} hg/ha** ({best_random_row['RMSE']/10_000:,.3f} tonnes/ha).
- On the more realistic future split, **{best_time_row['Model']}** achieved the lowest RMSE: **{best_time_row['RMSE']:,.2f} hg/ha**, MAE **{best_time_row['MAE']:,.2f} hg/ha**, and R² **{best_time_row['Test R2']:.3f}**.
- The difference between the best future-split and random-split RMSE is **{time_penalty:,.2f} hg/ha**. This shows why evaluation design matters: random splitting can mix nearby years across train and test and may give an optimistic estimate of future performance.
- The Decision Tree's train-versus-test R² gap demonstrates overfitting; the Random Forest reduces variance by averaging many trees.

### Limitations

- Rainfall and temperature are annual averages and do not describe growing-season timing or extreme weather.
- Pesticide usage is country-level and does not prove a causal effect on yield.
- Soil quality, irrigation, fertilizer, crop variety, farming practices, and economic conditions are absent.
- Countries and crops are unevenly represented, and historical relationships may change over time.
- The model estimates yield only when scenario inputs are supplied; it does not forecast future weather or pesticide use.

### Practical next steps

Add growing-season weather, soil, irrigation and fertilizer features; collect newer observations; evaluate crop-specific models; and use rolling-origin validation before considering real agricultural decisions.
'''))